In [22]:
import os
from pathlib import Path

from dotenv import load_dotenv


def find_project_root() -> Path:
    """Directory that contains pyproject.toml and deepAgentsDemo/."""
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "deepAgentsDemo").is_dir():
            return candidate
    raise FileNotFoundError(
        "Start Jupyter from the DeepAgents folder (the one that contains pyproject.toml)."
    )


PROJECT_ROOT = find_project_root()
env_file = PROJECT_ROOT / ".env"
loaded = load_dotenv(env_file, override=True)
print(f"Loaded .env from {env_file}: {loaded}")

groq_key = os.getenv("GROQ_API_KEY")
tavily_key = os.getenv("TAVILY_API_KEY")

if not groq_key or not tavily_key:
    raise ValueError(
        f"Missing keys after loading {env_file}. "
        "Confirm GROQ_API_KEY and TAVILY_API_KEY are saved in that file."
    )

os.environ["GROQ_API_KEY"] = groq_key
os.environ["TAVILY_API_KEY"] = tavily_key
print("API keys loaded.")
print("Project root:", PROJECT_ROOT)


Loaded .env from /Users/varun.patil/Desktop/DeepAgents/.env: True
API keys loaded.


In [23]:
# Tools - Search
from tavily import TavilyClient
from typing import Literal

tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
print("Tavily client ready.")


def web_search(
    query: str,
    max_results: int = 5,
    topic: Literal["general", "news", "finance"] = "general",
    include_raw_content: bool = False,
):
    """Run a web search using Tavily."""
    return tavily_client.search(
        query,
        max_results=max_results,
        include_raw_content=include_raw_content,
        topic=topic,
    )


Tavily client ready.


In [24]:
from langchain.chat_models import init_chat_model

# qwen/qwen3-32b is not available on Groq; use a current model id
model = init_chat_model(
    "qwen/qwen3.6-27b",
    model_provider="groq",
    api_key=os.environ["GROQ_API_KEY"],
)
print(model)


metadata={'lc_versions': {'langchain-core': '1.5.4', 'langchain': '1.3.15'}} client=<groq.resources.chat.completions.Completions object at 0x112305310> async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x1128cbd40> model_name='qwen/qwen3.6-27b' model_kwargs={} groq_api_key=SecretStr('**********')


In [25]:
from langchain.agents import create_agent



In [26]:
# Create a Deep Agent
from deepagents import create_deep_agent

deep_agent = create_deep_agent(
    model=model,
    tools=[web_search],
    system_prompt="Act as a Researcher",
)
print(deep_agent)


In [27]:
simple_agent = create_agent(
    model=model,
    tools=[web_search],

)
print(simple_agent)

In [8]:
result = deep_agent.invoke({"messages":[{"role":"user","content":"What is LangGraph?"}]})
print(result)

{'messages': [HumanMessage(content='What is LangGraph?', additional_kwargs={}, response_metadata={}, id='a3f47edc-8b19-4674-98e8-934cef644103'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user is asking "What is LangGraph?".\nThis is a general knowledge question about a specific software library/framework.\nI should search the web to get the most up-to-date and accurate information about LangGraph, as it is a relatively new tool in the AI agent ecosystem.\n\nPlan:\n1.  Search for "LangGraph" to understand its definition, purpose, and key features.\n2.  Synthesize the findings to answer the user\'s question clearly.\n\nI will start by performing a web search.\n', 'tool_calls': [{'id': '7fdjb03yx', 'function': {'arguments': '{"query":"What is LangGraph?"}', 'name': 'web_search'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 140, 'prompt_tokens': 2617, 'total_tokens': 2757, 'completion_time': 0.266892994, 'completion_tokens_details

## State Backend


In [28]:
from deepagents import create_deep_agent
from deepagents.backends import StateBackend
from langgraph.checkpoint.memory import MemorySaver

# Pass the Groq model object — a bare "qwen/..." string can't infer provider
AGENT = create_deep_agent(
    model=model,
    backend=StateBackend(),
    checkpointer=MemorySaver(),
)

result = AGENT.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "1) What is LangGraph? "
                    "2) Create a Todo file named todo1.txt "
                    "3) Edit it "
                    "4) Confirm you have done it and show the file contents"
                ),
            }
        ]
    },
    config={"configurable": {"thread_id": "state-backend-demo"}},
)

final = result["messages"][-1]
print(final.content)
print("\nFiles in state:", list(result.get("files", {}).keys()))


The file **`/home/user/todo1.txt`** has been successfully created and edited. As you can see:
- **Task 1** is marked as completed (`[x]`)
- **Tasks 2–4** remain pending (`[ ]`)

Files in state: ['/home/user/todo1.txt']


In [29]:
from pathlib import Path
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend

ROOT = PROJECT_ROOT / "deepAgentsDemo"

fs_agent = create_deep_agent(
    model=model,
    backend=FilesystemBackend(root_dir=ROOT, virtual_mode=True),
)
print(fs_agent)
print("Filesystem root:", ROOT)


Filesystem root: /Users/varun.patil/Desktop/DeepAgents/deepAgentsDemo


In [30]:
from pathlib import Path

agents_md_path = PROJECT_ROOT / "deepAgentsDemo" / "project" / "agents.md"
agents_md = agents_md_path.read_text(encoding="utf-8")
print(agents_md)


# Project Agents

## Researcher
Act as a careful researcher. Prefer web search for current facts.

## Todo Writer
Create and edit todo files clearly and concisely.



In [ ]:
from pathlib import Path
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from langgraph.checkpoint.memory import MemorySaver

ROOT = PROJECT_ROOT / "deepAgentsDemo"

# memory paths are virtual paths under the FilesystemBackend root
# (your file lives at deepAgentsDemo/project/agents.md)
agents = create_deep_agent(
    model=model,
    backend=FilesystemBackend(root_dir=ROOT, virtual_mode=True),
    memory=["/project/agents.md"],
    checkpointer=MemorySaver(),
)
print(agents)


Structured Output with SubAgents

In [ ]:
from pydantic import BaseModel,Field
from deepagents import create_deep_agent

class researchFindings(BaseModel):
    """Structured Findings from a task"""
    summary:str=Field(description="Summary of findings")
    findings:list[str]=Field()
